# Notebook: El mundo del Big Data a través del cine

Vamos a explorar los [datasets de IMDb](https://data.imdb.com/non-commercial-datasets/) con **PySpark** y **Spark SQL**. Usaremos datos reales para hacer consultas, revisar sus resultados y ver cómo Spark las ejecuta.

El notebook tiene dos partes: **configuración** y **consultas**. Necesitas tener instalados Java, Python, PySpark y JupyterLab, como se explica en las [instrucciones del taller](https://github.com/aidhog/big-data-cine/).

Recomiendo dejar los datos comprimidos (`.tsv.gz`), pero también puedes usar archivos `.tsv`. Spark los lee directamente. Los ejemplos ya preparados usan las tablas `title.basics`, `title.ratings`, `title.crew` y `name.basics`.

## Parte 1. Preparar el entorno

### A. Iniciar Spark
Ejecuta esta celda para crear una sesión local de Spark en tu equipo.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("big-data-cine")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 4.2.0


### B. Encontrar los archivos
Buscamos primero en `data/`, luego en la carpeta del notebook y finalmente en `../data/`. Si están los dos formatos, preferimos el comprimido. Ejecuta esta celda antes de avanzar.

In [2]:
from pathlib import Path

def imdb_file(name):
    for folder in (Path("data"), Path("."), Path("../data")):
        for extension in (".tsv.gz", ".tsv"):
            path = folder / f"{name}{extension}"
            if path.is_file():
                return str(path)
    raise FileNotFoundError(f"No encontré {name}.tsv.gz ni {name}.tsv")

paths = {
    name: imdb_file(name)
    for name in ("title.basics", "title.ratings", "title.crew", "name.basics")
}
for name, path in paths.items():
    print(name, "→", path)

title.basics → ..\data\title.basics.tsv.gz
title.ratings → ..\data\title.ratings.tsv.gz
title.crew → ..\data\title.crew.tsv.gz
name.basics → ..\data\name.basics.tsv.gz


### C. Leer los datos y definir los tipos

IMDb usa `\N` para indicar que falta un valor. No tenemos que hacer los _casts_ ahora, los podemos hacer en las consultas, pero así las consultas quedan más simples. Ejecuta esta celda antes de avanzar.

In [4]:
def read_imdb(name):
    return (
        spark.read
        .option("header", True)
        .option("sep", "\t")
        .option("nullValue", "\\N")
        .csv(paths[name])
    )

# Convertimos las columnas numéricas una vez en la definición de las tablas.
titles = read_imdb("title.basics").selectExpr(
    "tconst", "titleType", "primaryTitle", "originalTitle",
    "try_cast(isAdult AS INT) AS isAdult",
    "try_cast(startYear AS INT) AS startYear",
    "try_cast(endYear AS INT) AS endYear",
    "try_cast(runtimeMinutes AS INT) AS runtimeMinutes",
    "genres"
)
ratings = read_imdb("title.ratings").selectExpr(
    "tconst",
    "try_cast(averageRating AS DOUBLE) AS averageRating",
    "try_cast(numVotes AS BIGINT) AS numVotes"
)
crew = read_imdb("title.crew")
names = read_imdb("name.basics").selectExpr(
    "nconst", "primaryName",
    "try_cast(birthYear AS INT) AS birthYear",
    "try_cast(deathYear AS INT) AS deathYear",
    "primaryProfession", "knownForTitles"
)

for view, df in (("titles", titles), ("ratings", ratings),
                 ("crew", crew), ("names", names)):
    df.createOrReplaceTempView(view)

print("Tablas listas: titles, ratings, crew, names")
titles.printSchema()

Tablas listas: titles, ratings, crew, names
root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: integer (nullable = true)
 |-- genres: string (nullable = true)



## Parte 2. Consultas

En cada ejemplo puedes ejecutar la consulta, ver los resultados y examinar el plan con `explain()`. Cambia los filtros y vuelve a ejecutar para explorar los datos.

### Consulta 1. Películas de ciencia ficción bien evaluadas
Buscamos largometrajes (al menos `60` minutos) del género `Sci-Fi` o `Fantasy`, que no sean documentales (`Documentary`), estrenados entre `1960` y `2020`, con nota mayor que `8,0` y entre `10.000` y `100.000` votos. Ordenamos de mayor a menor nota.

In [16]:
%%time

consulta_1 = """
SELECT t.tconst, t.primaryTitle AS titulo, t.startYear AS anio,
       t.runtimeMinutes AS minutos, r.averageRating AS nota,
       r.numVotes AS votos
FROM titles t
JOIN ratings r ON t.tconst = r.tconst
WHERE t.titleType = 'movie'
  AND t.runtimeMinutes >= 60
  AND arrays_overlap( split(t.genres, ','), array('Sci-Fi', 'Fantasy') )
  AND NOT array_contains(split(t.genres, ','), 'Documentary')
  AND t.startYear BETWEEN 1960 AND 2020
  AND r.averageRating > 8.0
  AND r.numVotes BETWEEN 10000 AND 100000
ORDER BY nota DESC, votos DESC
"""
resultado_1 = spark.sql(consulta_1)
resultado_1.show(100, truncate=False)

+----------+-------------------------------------------------+----+-------+----+-----+
|tconst    |titulo                                           |anio|minutos|nota|votos|
+----------+-------------------------------------------------+----+-------+----+-----+
|tt14888874|Demon Slayer: Kimetsu no Yaiba - Mt. Natagumo Arc|2020|138    |8.7 |19053|
|tt0271383 |A Dog's Will                                     |2000|104    |8.6 |22266|
|tt8652818 |Violet Evergarden: The Movie                     |2020|140    |8.3 |15439|
|tt2358592 |Lucia                                            |2013|135    |8.3 |14041|
|tt9881586 |Rascal Does Not Dream of a Dreaming Girl         |2019|89     |8.3 |11238|
|tt0104860 |We Are Not Angels                                |1992|98     |8.3 |10328|
|tt8239946 |Tumbbad                                          |2018|108    |8.2 |78049|
|tt0070233 |Ivan Vasilyevich Changes His Profession          |1973|88     |8.2 |20850|
|tt2283748 |OMG: Oh My God!                

In [6]:
resultado_1.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (12)
+- Sort (11)
   +- Exchange (10)
      +- Project (9)
         +- BroadcastHashJoin Inner BuildRight (8)
            :- Project (3)
            :  +- Filter (2)
            :     +- Scan csv  (1)
            +- BroadcastExchange (7)
               +- Project (6)
                  +- Filter (5)
                     +- Scan csv  (4)


(1) Scan csv 
Output [6]: [tconst#117, titleType#118, primaryTitle#119, startYear#122, runtimeMinutes#124, genres#125]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/aidhog/Documents/Teaching/BigData/taller/data/title.basics.tsv.gz]
PushedFilters: [IsNotNull(titleType), IsNotNull(runtimeMinutes), IsNotNull(genres), IsNotNull(startYear), EqualTo(titleType,movie), IsNotNull(tconst)]
ReadSchema: struct<tconst:string,titleType:string,primaryTitle:string,startYear:string,runtimeMinutes:string,genres:string>

(2) Filter
Input [6]: [tconst#117, titleType#118, primaryTitle#119, startYear#122, runtimeMinutes#124

### Consulta 2. Documentales sobre Big Data
Buscamos documentales estrenados desde `2000` cuyo título contiene la frase “`Big Data`”, sin distinguir entre mayúsculas y minúsculas.

In [13]:
%%time

consulta_2 = """
SELECT t.tconst, t.primaryTitle AS titulo, t.startYear AS anho,
       t.runtimeMinutes AS minutos, r.averageRating AS nota,
       r.numVotes AS votos
FROM titles t
LEFT JOIN ratings r ON t.tconst = r.tconst
WHERE array_contains(split(t.genres, ','), 'Documentary')
  AND t.startYear >= 2000
  AND lower(t.primaryTitle) LIKE '%big data%'
ORDER BY anho DESC, titulo
"""
resultado_2 = spark.sql(consulta_2)
resultado_2.show(100, truncate=False)

+----------+-----------------------------------------------------------------------------+----+-------+----+-----+
|tconst    |titulo                                                                       |anho|minutos|nota|votos|
+----------+-----------------------------------------------------------------------------+----+-------+----+-----+
|tt26686517|Big Data Is Watching                                                         |2023|102    |NULL|NULL |
|tt32097705|George Orwell: Big Brother und Big Data                                      |2023|20     |NULL|NULL |
|tt41362427|Big Data: quand les politiques nous ciblent                                  |2022|NULL   |NULL|NULL |
|tt37164406|IA, BIG DATA ET NUMÉRIQUE: L'ALGORITHME DANS LA PEAU - Science En Direct 2021|2022|45     |NULL|NULL |
|tt19391230|Secrets of Big Data                                                          |2022|NULL   |7.6 |38   |
|tt16021676|Big Data                                                            

In [ ]:
resultado_2.explain(mode="formatted")

### Consulta 3. Películas dirigidas por Maite Alberdi

En este ejemplo, bucamos las películas dirigidas por Maite Alberdi.

In [8]:
%%time

consulta_3 = """
WITH directores AS (
    SELECT c.tconst, explode(split(c.directors, ',')) AS nconst
    FROM crew c
)
SELECT DISTINCT t.tconst, t.primaryTitle AS titulo, t.startYear AS anio,
       t.runtimeMinutes AS minutos, r.averageRating AS nota,
       r.numVotes AS votos
FROM directores d
JOIN names n ON d.nconst = n.nconst
JOIN titles t ON d.tconst = t.tconst
LEFT JOIN ratings r ON t.tconst = r.tconst
WHERE n.primaryName = 'Maite Alberdi'
  AND t.titleType = 'movie'
ORDER BY anio DESC, titulo
"""
resultado_3 = spark.sql(consulta_3)
resultado_3.show(100, truncate=False)

+----------+-----------------------------------------------+----+-------+----+-----+
|tconst    |titulo                                         |anio|minutos|nota|votos|
+----------+-----------------------------------------------+----+-------+----+-----+
|tt33565722|A Child of My Own                              |2026|96     |6.1 |1038 |
|tt31796256|In Her Place                                   |2024|96     |6.3 |7452 |
|tt24082488|The Eternal Memory                             |2023|85     |7.4 |4173 |
|tt21815036|Soy la Tierra. Historias desde el fin del mundo|2022|46     |7.6 |11   |
|tt11394298|The Mole Agent                                 |2020|84     |7.5 |10382|
|tt6618178 |The Grown-Ups                                  |2016|82     |7.5 |353  |
|tt4147684 |La Once                                        |2014|80     |7.5 |626  |
|tt2097257 |The Lifeguard                                  |2011|64     |6.9 |193  |
+----------+-----------------------------------------------+----+

In [ ]:
resultado_3.explain(mode="formatted")

### Consulta 4. Parejas de directores con varias películas destacadas

Buscamos pares de directores que hayan co-dirigido múltiples películas con nota mayor que 8,0 y al menos 10.000 votos. Ordenamos las parejas según el número de películas que cumplen esas condiciones, de mayor a menor.

Esta consulta es más exigente que las anteriores: puede tardar más.

In [15]:
%%time

consulta_4 = """
WITH peliculas_destacadas AS (
    SELECT t.tconst
    FROM titles t
    JOIN ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie'
      AND r.averageRating > 8.0
      AND r.numVotes >= 10000
),
directores AS (
    SELECT p.tconst, explode(split(c.directors, ',')) AS nconst
    FROM peliculas_destacadas p
    JOIN crew c ON p.tconst = c.tconst
),
parejas AS (
    SELECT a.nconst AS director_1, b.nconst AS director_2,
           a.tconst
    FROM directores a
    JOIN directores b ON a.tconst = b.tconst
    WHERE a.nconst < b.nconst
),
conteos AS (
    SELECT director_1, director_2,
           count(DISTINCT tconst) AS peliculas_destacadas
    FROM parejas
    GROUP BY director_1, director_2
    HAVING count(DISTINCT tconst) >= 2
)
SELECT n1.primaryName AS director_1,
       n2.primaryName AS director_2,
       c.peliculas_destacadas
FROM conteos c
JOIN names n1 ON c.director_1 = n1.nconst
JOIN names n2 ON c.director_2 = n2.nconst
ORDER BY c.peliculas_destacadas DESC, director_1, director_2
"""
resultado_4 = spark.sql(consulta_4)
resultado_4.show(100, truncate=False)

+-------------------------+----------+--------------------+
|director_1               |director_2|peliculas_destacadas|
+-------------------------+----------+--------------------+
|Ethan Coen               |Joel Coen |3                   |
|Anthony Russo            |Joe Russo |2                   |
|Elizabeth Chai Vasarhelyi|Jimmy Chin|2                   |
+-------------------------+----------+--------------------+

CPU times: total: 0 ns
Wall time: 1min 3s


In [ ]:
resultado_4.explain(mode="formatted")

### Consulta 5. ¡Tu turno!

Inventa una pregunta que se pueda responder con IMDb y escribe tu propia consulta SQL. Puedes usar las tablas `titles`, `ratings`, `crew` y `names` (o en caso de querer usar otras tablas, las puedes cargar arriba, exteniendo y corriendo de nuevo la parte 1.B).

Algunas ideas: ¿qué géneros tienen más películas?, ¿qué directores han trabajado en más títulos?, ¿cómo han cambiado las duraciones a lo largo de los años?, ¿qué películas cuentan con `Chile` en su título?

Se da una consulta base, pero puedes usar otra consulta dada antes como ejemplo. Adáptala y ejecuta las tres celdas.

In [ ]:
consulta_5 = """
-- Escribe aquí tu consulta.
-- Ejemplo para comenzar (puedes reemplazarlo):
SELECT titleType, count(*) AS cantidad
FROM titles
GROUP BY titleType
ORDER BY cantidad DESC
"""
resultado_5 = spark.sql(consulta_5)

In [ ]:
resultado_5.show(100, truncate=False)

In [ ]:
resultado_5.explain(mode="formatted")